Import dataset

In [1]:
import pandas as pd
df = pd.read_csv("assets/hospital_dataset.csv")
df.dtypes.reset_index()

,index,0
0,Transaction_ID,object
1,Patient_ID,object
2,Patient_Name,object
3,Age,int64
4,Gender,object
5,Blood_Type,object
6,Admission_Type,object
7,Department,object
8,Consultant_Doctor,object
9,Room_Number,int64


Data Prep

In [2]:
date_cols = ['Admission_Date', 'Discharge_Date', 'Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])


    df['Zip_Code'] = df['Zip_Code'].astype(str)

In [3]:
df.dtypes.reset_index()

,index,0
0,Transaction_ID,object
1,Patient_ID,object
2,Patient_Name,object
3,Age,int64
4,Gender,object
5,Blood_Type,object
6,Admission_Type,object
7,Department,object
8,Consultant_Doctor,object
9,Room_Number,int64


In [4]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go

# OVERVIEW

In [5]:
total_transactions = df['Transaction_ID'].nunique()
total_revenue = df['Total_Sales_Amount'].sum()
avg_billing = df['Billing_Amount'].mean()
avg_stay = df['Length_of_Stay'].mean()
avg_satisfaction = df['Patient_Satisfaction_Score'].mean()
emergency_pct = (df['Emergency_Case'] == 'Yes').mean() * 100

print(f"Total Transactions:         {total_transactions}")
print(f"Total Revenue:              ${total_revenue:,.2f}")
print(f"Average Billing Amount:    ${avg_billing:,.2f}")
print(f"Average Length of Stay:     {avg_stay:.2f} days")
print(f"Average Satisfaction:       {avg_satisfaction:.2f} / 10")
print(f"Emergency Case Rate:        {emergency_pct:.1f}%")

Total Transactions:         700
Total Revenue:              $22,052,423.53
Average Billing Amount:    $22,996.33
Average Length of Stay:     7.79 days
Average Satisfaction:       5.49 / 10
Emergency Case Rate:        47.9%


In [6]:

revenue_trend = df.groupby('Date')['Total_Sales_Amount'].sum().reset_index()

line_fig = px.area(
    revenue_trend, 
    x='Date', 
    y='Total_Sales_Amount',
    title='Hospital Daily Revenue Trend (Total Sales)',
    labels={'Total_Sales_Amount': 'Revenue ($)', 'Date': 'Transaction Date'},
    template='plotly_white'
)

line_fig.update_traces(line_color='#2ca02c', line_width=2)

line_fig.update_layout(
    hovermode='x unified',
    xaxis_title="Timeline",
    yaxis_title="Total Sales Amount ($)",
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1,
                     label="1m",
                     step="month",
                     stepmode="backward"),
                dict(count=6,
                     label="6m",
                     step="month",
                     stepmode="backward"),
                dict(count=1,
                     label="YTD",
                     step="year",
                     stepmode="todate"),
                dict(count=1,
                     label="1y",
                     step="year",
                     stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ), 

)

line_fig.show()

In [7]:
# SCATTER PLOT

df_3d = df.groupby(['Department', pd.Grouper(key='Date', freq='W')]).size().reset_index(name='Patient_Count')

scatter_fig = go.Figure(data=[go.Scatter3d(
    x=df_3d['Date'],
    y=df_3d['Department'],
    z=df_3d['Patient_Count'],
    mode='markers',
    marker=dict(
        size=7,
        color=df_3d['Patient_Count'],   
        colorscale='Viridis',           
        colorbar_title = 'Number of Patients',
        line_color='rgb(140, 140, 170)'
    )
)])

scatter_fig.update_layout(
    title='3D View: Weekly Patient Volume per Department',
    scene=dict(
        xaxis_title='Timeline',
        yaxis_title='Department',
        zaxis_title='Number of Patients'
    ),
    margin=dict(l=0, r=0, b=0, t=50),
)

scatter_fig.show()

In [8]:
# 3D SURFACE / 2D HEATMAP
z_data = pd.crosstab(df['Department'], df['Treatment_Plan'])

surface_heatmap_fig = go.Figure()

surface_heatmap_fig.add_trace(go.Surface(
    z=z_data.values, 
    x=z_data.columns, 
    y=z_data.index, 
    colorscale="Greens"
))

surface_heatmap_fig.update_layout(
    title='Department vs. Treatment Plan Analysis',
    width=800,
    height=800,
    autosize=False,
    margin=dict(t=0, b=0, l=0, r=0),
    template="plotly_white",
    scene=dict(
        xaxis_title='Treatment Plan',
        yaxis_title='Department',
        zaxis_title='Volume',
        aspectratio=dict(x=1, y=1, z=0.7),
        aspectmode="manual"
    ),
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{"type": "surface"}],
                    label="3D Surface",
                    method="restyle"
                ),
                dict(
                    args=[{"type": "heatmap"}],
                    label="2D Heatmap",
                    method="restyle"
                )
            ]),
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ]

)

surface_heatmap_fig.update_traces(contours_z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True))

surface_heatmap_fig.show()

In [9]:
status_counts = df['Payment_Status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']

bar_pie_fig = go.Figure()

bar_pie_fig.add_trace(go.Bar(
    x=status_counts['Status'],
    y=status_counts['Count'],
    marker_color=['#2E7D32', '#4CAF50', '#81C784'],
    visible=True
))


bar_pie_fig.add_trace(go.Pie(
    labels=status_counts['Status'],
    values=status_counts['Count'],
    marker=dict(colors=['#2E7D32', '#4CAF50', '#81C784']),
    visible=False
))

bar_pie_fig.update_layout(
    title='Patient Distribution by Payment Status',
    template='plotly_white',
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.1,
            y=1.15,
            showactive=True,
            buttons=list([
                dict(
                    label="Bar Chart",
                    method="update",
                    args=[{"visible": [True, False]}, 
                          {"yaxis": {"title": "Number of Patients"}, 
                           "xaxis": {"title": "Status"}}]
                ),
                dict(
                    label="Pie Chart",
                    method="update",
                    args=[{"visible": [False, True]}, 
                          {"yaxis": {"visible": False}, 
                           "xaxis": {"visible": False}}]
                )
            ]),
        )
    ]
)


bar_pie_fig.show()

In [10]:
method_counts = df['Payment_Method'].value_counts().reset_index()
method_counts.columns = ['Method', 'Count']

bar_pie_2_fig = go.Figure()

bar_pie_2_fig.add_trace(go.Bar(
    x=method_counts['Method'],
    y=method_counts['Count'],
    marker_color=["#1E5E21", "#288D2B",'#4CAF50',  '#81C784'],
    visible=True
))

bar_pie_2_fig.add_trace(go.Pie(
    labels=method_counts['Method'],
    values=method_counts['Count'],
    marker=dict(colors=["#1E5E21", "#288D2B",'#4CAF50',  '#81C784']),
    visible=False
))

bar_pie_2_fig.update_layout(
    title='Patient Distribution by Payment Method',
    template='plotly_white',
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.1,
            y=1.15,
            showactive=True,
            buttons=list([
                dict(
                    label="Bar Chart",
                    method="update",
                    args=[{"visible": [True, False]}, 
                          {"yaxis": {"title": "Number of Patients"}, 
                           "xaxis": {"title": "Status"}}]
                ),
                dict(
                    label="Pie Chart",
                    method="update",
                    args=[{"visible": [False, True]}, 
                          {"yaxis": {"visible": False}, 
                           "xaxis": {"visible": False}}]
                )
            ]),
        )
    ]
)

bar_pie_2_fig.show()

In [11]:

df['Month'] = df['Date'].dt.strftime('%B %Y')
months = sorted(df['Date'].unique())
month_labels = [d.strftime('%B %Y') for d in pd.to_datetime(months).unique()]

def get_sankey_data(filtered_df):
    # Flow 1: Admission_Type -> Department
    flow1 = filtered_df.groupby(['Admission_Type', 'Department']).size().reset_index(name='value')
    flow1.columns = ['source', 'target', 'value']
    
    # Flow 2: Department -> Payment_Status
    flow2 = filtered_df.groupby(['Department', 'Payment_Status']).size().reset_index(name='value')
    flow2.columns = ['source', 'target', 'value']
    
    full_flow = pd.concat([flow1, flow2])
    
    # Unique labels for nodes
    labels = list(pd.unique(full_flow[['source', 'target']].values.ravel('K')))
    mapping = {label: i for i, label in enumerate(labels)}
    
    return labels, {
        'source': full_flow['source'].map(mapping).tolist(),
        'target': full_flow['target'].map(mapping).tolist(),
        'value': full_flow['value'].tolist()
    }

# 2. Create Initial Figure
initial_month = month_labels[0]
labels, data = get_sankey_data(df[df['Month'] == initial_month])

sankey_fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20, line=dict(color="black", width=0.5),
        label=labels,
        color="#2E7D32" # Hospital Green
    ),
    link=dict(
        source=data['source'],
        target=data['target'],
        value=data['value'],
        color="rgba(76, 175, 80, 0.4)" # Transparent Green
    )
)])

# 3. Add Slider Logic (Frames)
frames = []
for m in month_labels:
    l, d = get_sankey_data(df[df['Month'] == m])
    frames.append(go.Frame(
        data=[go.Sankey(
            node=dict(label=l, color="#2E7D32"),
            link=dict(source=d['source'], target=d['target'], value=d['value'])
        )],
        name=m
    ))

sankey_fig.frames = frames

# 4. Configure Slider Layout
sliders = [dict(
    steps=[dict(
        method="animate",
        args=[[m], dict(mode="immediate", frame=dict(duration=300, redraw=True), transition=dict(duration=0))],
        label=m
    ) for m in month_labels],
    active=0,
    currentvalue={"prefix": "Selected Month: "},
    pad={"t": 50}
)]

sankey_fig.update_layout(
    title_text="Patient Flow: Admission → Department → Payment Status",
    font_size=12,
    template="plotly_white",
    sliders=sliders,
    height=600
)

sankey_fig.show()

# PATIENTS DASHBOARD

In [12]:
# STACKED BAR CHART

patient_trend = df.groupby(['Date', 'Admission_Type'])['Patient_ID'].nunique().reset_index()
patient_trend.columns = ['Date', 'Admission_Type', 'Patient_Count']


stacked_bar_fig = px.bar(
    patient_trend, 
    x='Date', 
    y='Patient_Count',
    color='Admission_Type',
    title='Hospital Patient Volume Trend Per Admission Type',
    template='plotly_white',
    color_discrete_sequence=["#1E5E21", "#288D2B",'#4CAF50',  '#81C784'] 
)

stacked_bar_fig.update_layout(
    hovermode='x unified',
    xaxis_title="Timeline",
    yaxis_title="Total Patients",
    bargap=0.1, 
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    ),
)

stacked_bar_fig.show()

In [13]:
# HISTOGRAM
hist_fig = px.histogram(
    df, 
    x="Age", 
    nbins=30,
    title="Age Distribution of Patients",
    template='plotly_white',
    color_discrete_sequence=['#076821'] 
)

hist_fig.update_layout(
    xaxis_title="Patient Age",
    yaxis_title="Frequency",
    hovermode='x unified',
    bargap=0.05,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type="linear"
    )
)

hist_fig.update_traces(
    marker_line_color='white',
    marker_line_width=1,
    opacity=0.85
)

hist_fig.show()

In [14]:
# TREE MAP/ HEATMAP
grouped = df.groupby(['Gender', 'Bed_Category'])['Patient_ID'].nunique().reset_index()
grouped.columns = ['Gender', 'Bed_Category', 'Count']

all_labels = ["Total Patients"] + grouped['Bed_Category'].unique().tolist() + grouped['Gender'].tolist()
all_parents = [""] + ["Total Patients"] * len(grouped['Bed_Category'].unique()) + grouped['Bed_Category'].tolist()
all_values = [grouped['Count'].sum()] + grouped.groupby('Bed_Category')['Count'].sum().tolist() + grouped['Count'].tolist()

heatmap_data = grouped.pivot(index='Gender', columns='Bed_Category', values='Count')

tree_heat_fig = go.Figure()

tree_heat_fig.add_trace(go.Treemap(
    labels=all_labels,
    parents=all_parents,
    values=all_values,
    branchvalues="total",
    marker=dict(colorscale='Greens'),
    visible=True,
    name="Treemap"
))

tree_heat_fig.add_trace(go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='Greens',
    visible=False,
    name="Heatmap",
    text=heatmap_data.values,
    texttemplate="%{text}", 
))

tree_heat_fig.update_layout(
    title='Patient Distribution: Gender vs. Bed Category',
    template='plotly_white',
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.01,
            y=1.15,
            showactive=True,
            buttons=list([
                dict(
                    label="Treemap View",
                    method="update",
                    args=[{"visible": [True, False]}, 
                          {"title": "Hierarchy: Bed Category & Gender"}]
                ),
                dict(
                    label="HeatMap View",
                    method="update",
                    args=[{"visible": [False, True]}, 
                          {"title": "Intensity: Gender vs. Bed Category"}]
                ),
            ]),
        )
    ]
)

tree_heat_fig.show()

In [15]:
# GAUGE
gauge_fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = avg_satisfaction,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': "Avg. Patient Satisfaction", 'font': {'size': 24, 'color': '#076821'}},
    gauge = {
        'axis': {'range': [0, 10], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': "#076821"}, 
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 5], 'color': '#FFCCCC'},   
            {'range': [5, 8], 'color': '#E8F5E9'},   
            {'range': [8, 10], 'color': '#A5D6A7'}  
        ],
        'threshold': {
            'line': {'color': "black", 'width': 4},
            'thickness': 0.75,
            'value': 9.0 # Target Score
        }
    }
))

gauge_fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font={'color': "#076821", 'family': "Arial"},
    height=400
)

gauge_fig.show()

In [16]:
emergency_cases = (df['Emergency_Case'] == 'Yes').sum()
follow_up_needed = (df['Follow_Up_Required'] == 'Yes').sum()

emergency_rate = (df['Emergency_Case'] == 'Yes').mean() * 100
follow_up_rate = (df['Follow_Up_Required'] == 'Yes').mean() * 100

print(f"Total Emergency Cases {emergency_cases}")
print(f"Emergency Case Rate {emergency_rate:.1f}%")
print(f"Total Follow-Ups Required {follow_up_needed}")
print(f"Follow-Up Required Rate {follow_up_rate:.1f}%")

Total Emergency Cases 335
Emergency Case Rate 47.9%
Total Follow-Ups Required 347
Follow-Up Required Rate 49.6%


# DOCTORS

In [17]:
total_doctors = df['Consultant_Doctor'].nunique()
avg_patients_per_doc = df.shape[0] / total_doctors
peak_workload = df['Consultant_Doctor'].value_counts().max()

print(f"Total Doctors:              {total_doctors}")
print(f"Avg Patients per Doctor:    {avg_patients_per_doc:.1f}")
print(f"Peak Workload (Max):        {peak_workload}")

Total Doctors:              8
Avg Patients per Doctor:    87.5
Peak Workload (Max):        102


In [18]:
def create_doctor_dropdown(df):
    doctors = sorted(df['Consultant_Doctor'].unique())
    dropdown_buttons = [
        dict(
            label="All Doctors",
            method="restyle",
            args=[{"visible": [True] * len(doctors)}]
        )
    ]

    for doctor in doctors:
        dropdown_buttons.append(dict(
            label=doctor,
            method="restyle",
            args=[{"visible": [d == doctor for d in doctors]}]
        )
    )
    return dropdown_buttons

In [19]:
# SCATTER PLOT
df['Year'] = pd.to_datetime(df['Admission_Date']).dt.year

fig_scatter = px.scatter(
    df, 
    x="Billing_Amount", 
    y="Patient_Satisfaction_Score", 
    animation_frame="Year", 
    animation_group="Consultant_Doctor",
    size="Length_of_Stay", 
    color="Department", 
    hover_name="Consultant_Doctor",
    log_x=True, 
    size_max=55, 
    range_x=[df['Billing_Amount'].min(), df['Billing_Amount'].max()], 
    range_y=[0, 10],
    title="Doctor Analysis: Satisfaction vs. Billing",
    template="plotly_white",
    color_discrete_sequence=['#076821', '#228B22', '#3CB371', '#66BB6A', '#81C784', '#A5D6A7', '#C8E6C9']
)

fig_scatter.show()


In [20]:
doctor_counts = df.groupby(['Consultant_Doctor']).size().reset_index(name='Patient_Count')

def create_doctor_dropdown(df):
    doctors = sorted(df['Consultant_Doctor'].unique())
    dropdown_buttons = [
        dict(
            label="All Doctors",
            method="restyle",
            args=[{"visible": [True] * len(doctors)}]
        )
    ]

    for doctor in doctors:
        dropdown_buttons.append(dict(
            label=doctor,
            method="restyle",
            args=[{"visible": [d == doctor for d in doctors]}]
        )
    )
    return dropdown_buttons

dropdown_buttons = create_doctor_dropdown(df)
fig_bar = px.bar(
    doctor_counts, 
    x="Consultant_Doctor", 
    y="Patient_Count", 
    color="Consultant_Doctor",
    hover_name="Consultant_Doctor",
    title="Patient Volume per Consultant Doctor",
    template="plotly_white",
    color_discrete_sequence=['#076821', '#228B22', '#3CB371', '#66BB6A', '#81C784', '#A5D6A7', '#C8E6C9']
)


fig_bar.update_layout(
    margin=dict(l=50, r=50, t=100, b=50),
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0.1, xanchor="left",
            y=1.15, yanchor="top",
        ),
    ]
)

fig_bar.show()

In [21]:
# SUNBURST
fig_sunburst = px.sunburst(
    df, 
    path=['Department', 'Consultant_Doctor'], 
    values='Billing_Amount', 
    color='Department',
    color_discrete_sequence=['#076821', '#1B5E20', '#2E7D32', '#388E3C', '#4CAF50', '#81C784'],
    title="Consultant Doctor Hierarchy by Department",
    template="plotly_white"
)

doctors = sorted(df['Consultant_Doctor'].unique())
dropdown_buttons = [
    dict(
        label="All Doctors",
        method="restyle",
        args=[{"visible": [True]}]
    )
]

for doctor in doctors:
    dropdown_buttons.append(dict(
        label=doctor,
        method="restyle",
        args=[{"visible": [True]}] 
    ))

fig_sunburst.update_layout(
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0, xanchor="left",
            y=1, yanchor="top",
        ),
    ],
    margin=dict(t=60, l=20, r=20, b=20)
)

fig_sunburst.update_traces(
    hovertemplate='<b>%{label}</b><br>Total Billing: $%{value:,.2f}'
)

fig_sunburst.show()

In [22]:
# TABLE
doc_table_df = df.groupby('Consultant_Doctor').agg({
    'Patient_ID': 'count',
    'Patient_Satisfaction_Score': 'mean',
    'Billing_Amount': 'sum',
    'Department': 'first'
}).reset_index().rename(columns={'Patient_ID': 'Total_Patients'})

doc_table_df = doc_table_df.sort_values('Consultant_Doctor')
doctors = doc_table_df['Consultant_Doctor'].tolist()

fig_table = go.Figure()

fig_table.add_trace(
    go.Table(
        header=dict(
            values=["<b>Consultant Doctor</b>", "<b>Department</b>", "<b>Patients</b>", "<b>Avg Satisfaction</b>", "<b>Total Revenue</b>"],
            fill_color='#076821',
            align='left',
            font=dict(color='white', size=13, family="Arial"),
            height=35
        ),
        cells=dict(
            values=[
                doc_table_df['Consultant_Doctor'],
                doc_table_df['Department'],
                doc_table_df['Total_Patients'],
                doc_table_df['Patient_Satisfaction_Score'].round(2),
                doc_table_df['Billing_Amount'].apply(lambda x: f"${x:,.2f}")
            ],
            fill_color='#F1F8E9',
            align='left',
            font=dict(color='#076821', size=12),
            height=30
        )
    )
)

fig_table.update_layout(
    title="Consultant Doctor Clinical Performance Directory",
    margin=dict(t=100, l=20, r=20, b=20)
)

fig_table.show()

In [23]:
# POLAR
doctor_counts = df.groupby(['Consultant_Doctor']).size().reset_index(name='Patient_Count')

fig_polar = px.bar_polar(
    doctor_counts, 
    r="Patient_Count", 
    theta="Consultant_Doctor",
    color="Consultant_Doctor",
    template="plotly_white",
    color_discrete_sequence=['#076821', '#228B22', '#3CB371', '#66BB6A', '#81C784', '#A5D6A7', '#C8E6C9'],
    title="Consultant Doctor Workload Distribution"
)

fig_polar.update_layout(
    margin=dict(t=100, b=50, l=50, r=50),
)

fig_polar.show()